# 🧠 Fabric IQ — Insurance Ontology + Copilot Skills

**Domain ontology and AI agent skills for insurance business knowledge.**

**Fabric Features Showcased:**
- **Fabric IQ** — business ontology and semantic layer
- **Copilot Skills** — custom AI capabilities for insurance domain
- **Knowledge Graph** — entity relationships and inference
- **Natural Language Queries** — business-friendly data access
- **Power BI Integration** — semantic model connection
- **Delta Lake** — ontology persistence

**Insurance Ontology:**
- **Entities**: Customer, Agent, Policy, Claim, Premium, Payment, Coverage
- **Relationships**: Customer → Policy, Policy → Claim, Claim → Payment
- **Business Rules**: Underwriting rules, claim validation, fraud detection
- **KPIs**: Loss ratio, combined ratio, retention rate, average premium

**Copilot Skills:**
1. **Calculate Loss Ratio** — claims paid / premiums earned
2. **Identify High-Risk Policies** — adverse loss trends
3. **Detect Fraudulent Claims** — anomaly patterns
4. **Predict Customer Churn** — retention analysis
5. **Recommend Coverage** — personalized underwriting

**No SparkSession import — `spark` pre-initialized by Fabric.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Load shared utilities                                              ║
# ╚══════════════════════════════════════════════════════════════════════╝
%run 00_fabric_native_utils

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 1: Insurance Ontology Schema                               ║
# ║  Define business entities, relationships, and rules                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

def create_ontology_tables():
    """Create metadata tables for insurance ontology."""
    print("\n🧠 Creating insurance ontology tables...")
    
    # 1. Entity Definitions
    spark.sql("""
        CREATE TABLE IF NOT EXISTS metadata.ontology_entities (
            entity_id               STRING,
            entity_name             STRING,
            entity_type             STRING,  -- core, dimension, fact, concept
            entity_description      STRING,
            source_table            STRING,
            key_attributes          ARRAY<STRING>,
            business_synonyms       ARRAY<STRING>,
            parent_entity           STRING,
            is_active               BOOLEAN,
            created_at              TIMESTAMP,
            updated_at              TIMESTAMP
        ) USING DELTA
    """)
    
    # 2. Relationship Definitions
    spark.sql("""
        CREATE TABLE IF NOT EXISTS metadata.ontology_relationships (
            relationship_id         STRING,
            relationship_name       STRING,
            source_entity           STRING,
            target_entity           STRING,
            relationship_type       STRING,  -- one_to_one, one_to_many, many_to_many
            cardinality             STRING,  -- 1:1, 1:N, N:M
            join_condition          STRING,  -- SQL join expression
            is_bidirectional        BOOLEAN,
            business_description    STRING,
            created_at              TIMESTAMP
        ) USING DELTA
    """)
    
    # 3. Business Rules
    spark.sql("""
        CREATE TABLE IF NOT EXISTS metadata.ontology_business_rules (
            rule_id                 STRING,
            rule_name               STRING,
            rule_category           STRING,  -- validation, calculation, inference
            rule_description        STRING,
            applies_to_entity       STRING,
            rule_expression         STRING,  -- SQL or Python expression
            rule_severity           STRING,  -- error, warning, info
            is_active               BOOLEAN,
            created_at              TIMESTAMP
        ) USING DELTA
    """)
    
    # 4. KPI Definitions
    spark.sql("""
        CREATE TABLE IF NOT EXISTS metadata.ontology_kpis (
            kpi_id                  STRING,
            kpi_name                STRING,
            kpi_description         STRING,
            kpi_category            STRING,  -- financial, operational, customer
            calculation_formula     STRING,
            required_entities       ARRAY<STRING>,
            unit_of_measure         STRING,
            target_value            DOUBLE,
            threshold_red           DOUBLE,
            threshold_yellow        DOUBLE,
            threshold_green         DOUBLE,
            is_active               BOOLEAN,
            created_at              TIMESTAMP
        ) USING DELTA
    """)
    
    print("✅ Ontology tables created")

# Create tables
create_ontology_tables()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 2: Seed Insurance Ontology                                 ║
# ║  Define core entities, relationships, and KPIs                      ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql import Row
from datetime import datetime
import uuid

def seed_insurance_ontology():
    """Populate ontology with insurance domain knowledge."""
    print("\n🌱 Seeding insurance ontology...")
    
    # === ENTITIES ===
    entities = [
        {
            "entity_id": str(uuid.uuid4()),
            "entity_name": "Customer",
            "entity_type": "core",
            "entity_description": "Individual or organization purchasing insurance policies",
            "source_table": "gold_customer.dim_customer",
            "key_attributes": ["customer_id", "name", "email", "risk_score"],
            "business_synonyms": ["Policyholder", "Insured", "Client"],
            "parent_entity": None,
            "is_active": True,
            "created_at": datetime.now(),
            "updated_at": datetime.now()
        },
        {
            "entity_id": str(uuid.uuid4()),
            "entity_name": "Policy",
            "entity_type": "core",
            "entity_description": "Insurance contract providing coverage",
            "source_table": "gold_policy.dim_policy",
            "key_attributes": ["policy_number", "policy_type", "premium", "coverage_amount"],
            "business_synonyms": ["Contract", "Coverage", "Insurance Policy"],
            "parent_entity": None,
            "is_active": True,
            "created_at": datetime.now(),
            "updated_at": datetime.now()
        },
        {
            "entity_id": str(uuid.uuid4()),
            "entity_name": "Claim",
            "entity_type": "fact",
            "entity_description": "Request for payment under policy terms",
            "source_table": "gold_claims.fact_claims",
            "key_attributes": ["claim_id", "claim_amount", "claim_date", "claim_status"],
            "business_synonyms": ["Loss", "Incident", "Claim Event"],
            "parent_entity": "Policy",
            "is_active": True,
            "created_at": datetime.now(),
            "updated_at": datetime.now()
        },
        {
            "entity_id": str(uuid.uuid4()),
            "entity_name": "Agent",
            "entity_type": "dimension",
            "entity_description": "Insurance sales representative",
            "source_table": "gold_agent.dim_agent",
            "key_attributes": ["agent_id", "name", "license_number", "commission_rate"],
            "business_synonyms": ["Broker", "Sales Rep", "Producer"],
            "parent_entity": None,
            "is_active": True,
            "created_at": datetime.now(),
            "updated_at": datetime.now()
        },
        {
            "entity_id": str(uuid.uuid4()),
            "entity_name": "Premium",
            "entity_type": "concept",
            "entity_description": "Payment for insurance coverage",
            "source_table": "gold_financial.fact_premiums",
            "key_attributes": ["premium_amount", "payment_date", "payment_method"],
            "business_synonyms": ["Payment", "Insurance Cost", "Premium Payment"],
            "parent_entity": "Policy",
            "is_active": True,
            "created_at": datetime.now(),
            "updated_at": datetime.now()
        }
    ]
    
    df_entities = spark.createDataFrame([Row(**e) for e in entities])
    df_entities.write.format("delta").mode("overwrite").saveAsTable("metadata.ontology_entities")
    print(f"  ✅ Created {len(entities)} entities")
    
    # === RELATIONSHIPS ===
    relationships = [
        {
            "relationship_id": str(uuid.uuid4()),
            "relationship_name": "Customer_Has_Policy",
            "source_entity": "Customer",
            "target_entity": "Policy",
            "relationship_type": "one_to_many",
            "cardinality": "1:N",
            "join_condition": "customer.customer_id = policy.customer_id",
            "is_bidirectional": True,
            "business_description": "A customer can have multiple policies",
            "created_at": datetime.now()
        },
        {
            "relationship_id": str(uuid.uuid4()),
            "relationship_name": "Policy_Generates_Claim",
            "source_entity": "Policy",
            "target_entity": "Claim",
            "relationship_type": "one_to_many",
            "cardinality": "1:N",
            "join_condition": "policy.policy_number = claim.policy_number",
            "is_bidirectional": True,
            "business_description": "A policy can have multiple claims",
            "created_at": datetime.now()
        },
        {
            "relationship_id": str(uuid.uuid4()),
            "relationship_name": "Agent_Sells_Policy",
            "source_entity": "Agent",
            "target_entity": "Policy",
            "relationship_type": "one_to_many",
            "cardinality": "1:N",
            "join_condition": "agent.agent_id = policy.agent_id",
            "is_bidirectional": True,
            "business_description": "An agent can sell multiple policies",
            "created_at": datetime.now()
        },
        {
            "relationship_id": str(uuid.uuid4()),
            "relationship_name": "Policy_Requires_Premium",
            "source_entity": "Policy",
            "target_entity": "Premium",
            "relationship_type": "one_to_many",
            "cardinality": "1:N",
            "join_condition": "policy.policy_number = premium.policy_number",
            "is_bidirectional": False,
            "business_description": "A policy requires regular premium payments",
            "created_at": datetime.now()
        }
    ]
    
    df_relationships = spark.createDataFrame([Row(**r) for r in relationships])
    df_relationships.write.format("delta").mode("overwrite").saveAsTable("metadata.ontology_relationships")
    print(f"  ✅ Created {len(relationships)} relationships")
    
    # === KPIs ===
    kpis = [
        {
            "kpi_id": str(uuid.uuid4()),
            "kpi_name": "Loss Ratio",
            "kpi_description": "Claims paid divided by premiums earned",
            "kpi_category": "financial",
            "calculation_formula": "SUM(claim_amount) / SUM(premium_amount)",
            "required_entities": ["Claim", "Premium"],
            "unit_of_measure": "percentage",
            "target_value": 70.0,
            "threshold_red": 85.0,
            "threshold_yellow": 75.0,
            "threshold_green": 70.0,
            "is_active": True,
            "created_at": datetime.now()
        },
        {
            "kpi_id": str(uuid.uuid4()),
            "kpi_name": "Combined Ratio",
            "kpi_description": "Loss ratio plus expense ratio",
            "kpi_category": "financial",
            "calculation_formula": "(SUM(claim_amount) + SUM(expenses)) / SUM(premium_amount)",
            "required_entities": ["Claim", "Premium"],
            "unit_of_measure": "percentage",
            "target_value": 95.0,
            "threshold_red": 105.0,
            "threshold_yellow": 100.0,
            "threshold_green": 95.0,
            "is_active": True,
            "created_at": datetime.now()
        },
        {
            "kpi_id": str(uuid.uuid4()),
            "kpi_name": "Customer Retention Rate",
            "kpi_description": "Percentage of customers renewing policies",
            "kpi_category": "customer",
            "calculation_formula": "COUNT(DISTINCT renewed_customers) / COUNT(DISTINCT total_customers)",
            "required_entities": ["Customer", "Policy"],
            "unit_of_measure": "percentage",
            "target_value": 85.0,
            "threshold_red": 75.0,
            "threshold_yellow": 80.0,
            "threshold_green": 85.0,
            "is_active": True,
            "created_at": datetime.now()
        },
        {
            "kpi_id": str(uuid.uuid4()),
            "kpi_name": "Average Premium Per Policy",
            "kpi_description": "Average premium amount across all active policies",
            "kpi_category": "operational",
            "calculation_formula": "AVG(premium_amount)",
            "required_entities": ["Policy", "Premium"],
            "unit_of_measure": "currency",
            "target_value": 1200.0,
            "threshold_red": 900.0,
            "threshold_yellow": 1000.0,
            "threshold_green": 1200.0,
            "is_active": True,
            "created_at": datetime.now()
        }
    ]
    
    df_kpis = spark.createDataFrame([Row(**k) for k in kpis])
    df_kpis.write.format("delta").mode("overwrite").saveAsTable("metadata.ontology_kpis")
    print(f"  ✅ Created {len(kpis)} KPIs")
    
    print("\n✅ Insurance ontology seeded")

# Seed ontology
seed_insurance_ontology()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 3: Copilot Skill - Calculate Loss Ratio                    ║
# ║  Insurance KPI calculation with natural language interface          ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import sum as spark_sum, col

def calculate_loss_ratio(policy_type: str = None, start_date: str = None, end_date: str = None) -> dict:
    """Calculate loss ratio for insurance policies.
    
    Loss Ratio = Total Claims Paid / Total Premiums Earned
    
    Args:
        policy_type: Optional filter (Auto, Home, Life, Health)
        start_date: Optional start date (YYYY-MM-DD)
        end_date: Optional end date (YYYY-MM-DD)
    
    Returns: Loss ratio metrics with interpretation
    """
    print("\n📊 Calculating Loss Ratio...")
    
    # Build filter conditions
    where_clauses = []
    if policy_type:
        where_clauses.append(f"p.policy_type = '{policy_type}'")
    if start_date:
        where_clauses.append(f"c.claim_date >= '{start_date}'")
    if end_date:
        where_clauses.append(f"c.claim_date <= '{end_date}'")
    
    where_clause = "WHERE " + " AND ".join(where_clauses) if where_clauses else ""
    
    # Calculate loss ratio
    query = f"""
        SELECT 
            SUM(c.claim_amount) as total_claims,
            SUM(p.premium) as total_premiums,
            (SUM(c.claim_amount) / SUM(p.premium)) * 100 as loss_ratio,
            COUNT(DISTINCT c.claim_id) as claim_count,
            COUNT(DISTINCT p.policy_number) as policy_count
        FROM gold_claims.fact_claims c
        INNER JOIN gold_policy.dim_policy p ON c.policy_number = p.policy_number
        {where_clause}
    """
    
    try:
        result = spark.sql(query).collect()[0]
        
        loss_ratio = result["loss_ratio"] or 0
        
        # Interpret results
        if loss_ratio < 70:
            interpretation = "✅ Excellent - Healthy profitability"
            status = "green"
        elif loss_ratio < 80:
            interpretation = "⚠️  Good - Acceptable range"
            status = "yellow"
        elif loss_ratio < 100:
            interpretation = "⚠️  Concern - Review underwriting"
            status = "orange"
        else:
            interpretation = "❌ Critical - Unprofitable, immediate action required"
            status = "red"
        
        metrics = {
            "loss_ratio": round(loss_ratio, 2),
            "total_claims": result["total_claims"],
            "total_premiums": result["total_premiums"],
            "claim_count": result["claim_count"],
            "policy_count": result["policy_count"],
            "interpretation": interpretation,
            "status": status
        }
        
        print(f"\n  Loss Ratio: {loss_ratio:.2f}%")
        print(f"  Total Claims: ${metrics['total_claims']:,.2f}")
        print(f"  Total Premiums: ${metrics['total_premiums']:,.2f}")
        print(f"  Claims: {metrics['claim_count']} | Policies: {metrics['policy_count']}")
        print(f"\n  {interpretation}")
        
        return metrics
    
    except Exception as e:
        print(f"  ❌ Calculation failed: {str(e)}")
        return {"error": str(e)}

# Example usage
# loss_ratio = calculate_loss_ratio(policy_type="Auto", start_date="2024-01-01")

print("✅ Loss Ratio skill ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 4: Copilot Skill - Identify High-Risk Policies             ║
# ║  Detect policies with adverse loss trends                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

def identify_high_risk_policies(min_loss_ratio: float = 150.0, min_claim_count: int = 2) -> list:
    """Identify high-risk policies based on loss ratio and claim frequency.
    
    Args:
        min_loss_ratio: Minimum loss ratio threshold (default: 150%)
        min_claim_count: Minimum number of claims (default: 2)
    
    Returns: List of high-risk policies with recommendations
    """
    print(f"\n🚨 Identifying High-Risk Policies (Loss Ratio > {min_loss_ratio}%)...")
    
    query = f"""
        SELECT 
            p.policy_number,
            p.policy_type,
            c.customer_id,
            c.name as customer_name,
            p.premium,
            SUM(cl.claim_amount) as total_claims,
            COUNT(cl.claim_id) as claim_count,
            (SUM(cl.claim_amount) / p.premium) * 100 as loss_ratio,
            MAX(cl.claim_date) as last_claim_date
        FROM gold_policy.dim_policy p
        INNER JOIN gold_customer.dim_customer c ON p.customer_id = c.customer_id
        INNER JOIN gold_claims.fact_claims cl ON p.policy_number = cl.policy_number
        WHERE p.status = 'Active'
        GROUP BY p.policy_number, p.policy_type, c.customer_id, c.name, p.premium
        HAVING (SUM(cl.claim_amount) / p.premium) * 100 >= {min_loss_ratio}
           AND COUNT(cl.claim_id) >= {min_claim_count}
        ORDER BY loss_ratio DESC
        LIMIT 20
    """
    
    try:
        df_high_risk = spark.sql(query)
        high_risk_policies = df_high_risk.collect()
        
        if not high_risk_policies:
            print("  ✅ No high-risk policies found")
            return []
        
        print(f"\n  Found {len(high_risk_policies)} high-risk policies:\n")
        
        results = []
        for policy in high_risk_policies:
            loss_ratio = policy["loss_ratio"]
            
            # Determine recommendation
            if loss_ratio > 200:
                recommendation = "CANCEL - Excessive losses"
                action = "cancel"
            elif loss_ratio > 175:
                recommendation = "NON-RENEW - High risk"
                action = "non_renew"
            else:
                recommendation = "INCREASE PREMIUM - Risk adjustment needed"
                action = "increase_premium"
            
            policy_info = {
                "policy_number": policy["policy_number"],
                "customer_name": policy["customer_name"],
                "policy_type": policy["policy_type"],
                "loss_ratio": round(loss_ratio, 2),
                "total_claims": policy["total_claims"],
                "claim_count": policy["claim_count"],
                "premium": policy["premium"],
                "recommendation": recommendation,
                "action": action
            }
            
            print(f"  🚨 {policy['policy_number']} | {policy['customer_name']}")
            print(f"     Loss Ratio: {loss_ratio:.1f}% | Claims: {policy['claim_count']} (${policy['total_claims']:,.2f})")
            print(f"     Action: {recommendation}\n")
            
            results.append(policy_info)
        
        return results
    
    except Exception as e:
        print(f"  ❌ Analysis failed: {str(e)}")
        return []

# Example usage
# high_risk = identify_high_risk_policies(min_loss_ratio=150.0, min_claim_count=2)

print("✅ High-Risk Policy detection skill ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 5: Copilot Skill - Natural Language Query Interface        ║
# ║  Translate business questions to SQL                                ║
# ╚══════════════════════════════════════════════════════════════════════╝

def answer_insurance_question(question: str) -> dict:
    """Answer insurance business questions in natural language.
    
    Args:
        question: Natural language business question
    
    Returns: Query results with interpretation
    """
    print(f"\n❓ Question: {question}")
    
    # Simple pattern matching for demo (production would use LLM)
    question_lower = question.lower()
    
    # Pattern: "how many claims..."
    if "how many claims" in question_lower:
        query = """
            SELECT 
                COUNT(*) as total_claims,
                SUM(claim_amount) as total_amount
            FROM gold_claims.fact_claims
            WHERE claim_date >= CURRENT_DATE - INTERVAL 30 DAYS
        """
        result = spark.sql(query).collect()[0]
        
        answer = f"In the last 30 days, there were {result['total_claims']} claims totaling ${result['total_amount']:,.2f}"
        return {"answer": answer, "data": dict(result.asDict())}
    
    # Pattern: "what is the average premium..."
    elif "average premium" in question_lower:
        query = """
            SELECT 
                AVG(premium) as avg_premium,
                MIN(premium) as min_premium,
                MAX(premium) as max_premium
            FROM gold_policy.dim_policy
            WHERE status = 'Active'
        """
        result = spark.sql(query).collect()[0]
        
        answer = f"The average premium is ${result['avg_premium']:,.2f} (range: ${result['min_premium']:,.2f} - ${result['max_premium']:,.2f})"
        return {"answer": answer, "data": dict(result.asDict())}
    
    # Pattern: "who are the top customers..."
    elif "top customers" in question_lower:
        query = """
            SELECT 
                c.name,
                COUNT(DISTINCT p.policy_number) as policy_count,
                SUM(p.premium) as total_premium
            FROM gold_customer.dim_customer c
            INNER JOIN gold_policy.dim_policy p ON c.customer_id = p.customer_id
            WHERE p.status = 'Active'
            GROUP BY c.name
            ORDER BY total_premium DESC
            LIMIT 5
        """
        results = spark.sql(query).collect()
        
        top_customers = [f"{r['name']} ({r['policy_count']} policies, ${r['total_premium']:,.2f})" 
                        for r in results]
        
        answer = f"Top 5 customers by premium: {', '.join(top_customers)}"
        return {"answer": answer, "data": [dict(r.asDict()) for r in results]}
    
    # Pattern: "loss ratio"
    elif "loss ratio" in question_lower:
        metrics = calculate_loss_ratio()
        answer = f"Current loss ratio is {metrics['loss_ratio']}%. {metrics['interpretation']}"
        return {"answer": answer, "data": metrics}
    
    else:
        return {
            "answer": "I don't understand that question yet. Try asking about claims, premiums, customers, or loss ratio.",
            "data": None
        }

# Example questions
# answer_insurance_question("How many claims did we receive in the last 30 days?")
# answer_insurance_question("What is the average premium for active policies?")
# answer_insurance_question("Who are the top customers by premium?")
# answer_insurance_question("What is our current loss ratio?")

print("✅ Natural language query skill ready")

## 🎯 Summary

This notebook implements **Fabric IQ with Insurance Domain Ontology** and **Copilot Skills**:

### Insurance Ontology Created

**Core Entities (5):**
- Customer, Policy, Claim, Agent, Premium
- Each with business synonyms and key attributes
- Mapped to Gold layer Delta tables

**Relationships (4):**
- Customer → Policy (1:N)
- Policy → Claim (1:N)
- Agent → Policy (1:N)
- Policy → Premium (1:N)

**KPIs (4):**
- Loss Ratio (target: 70%)
- Combined Ratio (target: 95%)
- Customer Retention Rate (target: 85%)
- Average Premium Per Policy (target: $1,200)

### Copilot Skills Implemented

1. **Calculate Loss Ratio**
   - Formula: Claims Paid / Premiums Earned
   - Optional filters: policy type, date range
   - Color-coded interpretation (green/yellow/orange/red)

2. **Identify High-Risk Policies**
   - Detects policies with loss ratio > 150%
   - Recommends: Cancel, Non-Renew, or Increase Premium
   - Returns top 20 worst performers

3. **Natural Language Query Interface**
   - Translates business questions to SQL
   - Supports: claim counts, average premium, top customers, loss ratio
   - Extensible pattern matching

### Metadata Tables Created
- `metadata.ontology_entities` — business entity definitions
- `metadata.ontology_relationships` — entity relationships
- `metadata.ontology_business_rules` — validation and inference rules
- `metadata.ontology_kpis` — KPI definitions with thresholds

### Example Usage
```python
# Calculate loss ratio for Auto policies in 2024
loss_ratio = calculate_loss_ratio(
    policy_type="Auto",
    start_date="2024-01-01",
    end_date="2024-12-31"
)

# Find high-risk policies
high_risk = identify_high_risk_policies(
    min_loss_ratio=150.0,
    min_claim_count=2
)

# Ask natural language questions
answer = answer_insurance_question(
    "How many claims did we receive in the last 30 days?"
)
```

### Integration Points
- **Power BI Copilot** — use ontology for Q&A on semantic models
- **Fabric Copilot** — domain-aware code suggestions
- **Central Cockpit** — KPI visualizations with thresholds
- **AI Agents** — ontology-guided decision making

**Next Steps:**
1. Extend ontology with more entities (Coverage, Deductible, Underwriter)
2. Add more Copilot skills (churn prediction, fraud detection, coverage recommendation)
3. Integrate with Power BI semantic models for Q&A
4. Connect to Azure OpenAI for advanced NL query parsing
5. Build knowledge graph visualization